# TCC2 - Previsao de Surtos de Dengue: Distrito Federal
**Autores:** Pedro Lucas Santana e Thiago Ribeiro Freitas  
**Curso:** Engenharia de Software - UnB  

Notebook focado nos dados do **Distrito Federal** (Brasilia, IBGE 5300108).  
Explora todas as camadas de dados (Bronze → Silver → Gold) e treina modelos XGBoost.

**Repositorio:** [MODELO-PREVISAO](https://github.com/modelo-previsao-dengue/MODELO-PREVISAO)

---
## 1. Setup: Montar Drive e Instalar Dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CANDIDATES = [
    '/content/drive/MyDrive/TCC2-DADOS',
    '/content/drive/MyDrive/TCC2-DADOS/data',
    '/content/drive/MyDrive/TCC2-DADOS/TCC2-DADOS',
]

DATA_DIR = None
for path in CANDIDATES:
    if os.path.exists(os.path.join(path, 'model_ready')):
        DATA_DIR = path
        break

if DATA_DIR is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive/TCC2-DADOS'):
        if 'model_ready' in dirs:
            DATA_DIR = root
            break

if DATA_DIR is None:
    raise FileNotFoundError('Pasta model_ready nao encontrada dentro de TCC2-DADOS.')

print(f'Dados encontrados em: {DATA_DIR}')

In [ ]:
!pip install -q xgboost optuna shap pyarrow

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import xgboost as xgb
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import json, glob
from pathlib import Path

base = Path(DATA_DIR)
IBGE_DF = '5300108'

print('Dependencias carregadas.')
print(f'Filtrando para: Brasilia/DF (IBGE {IBGE_DF})')

---
## 2. Verificar Dados no Drive

In [ ]:
print('=== Estrutura do Drive ===')
for folder in ['sinan', 'inmet', 'integrated', 'model_ready', 'reference']:
    p = base / folder
    if p.exists():
        size_mb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
        n_files = sum(1 for f in p.rglob('*') if f.is_file())
        print(f'  {folder:15s}  {size_mb:8.1f} MB  ({n_files} arquivos)')
    else:
        print(f'  {folder:15s}  NAO ENCONTRADO')

---
## 3. Explorar SINAN Silver (DF)
Notificacoes semanais observadas por municipio — camada **Silver** da arquitetura Medallion.

In [ ]:
sinan_silver = ds.dataset(
    f'{DATA_DIR}/sinan/silver/sinan_tcc2_v2/official_observed',
    format='parquet', partitioning='hive'
)

df_silver = sinan_silver.to_table(
    filter=ds.field('ibge_municipio') == IBGE_DF
).to_pandas()

print(f'SINAN Silver DF: {len(df_silver):,} linhas')
print(f'Periodo: {df_silver.ano.min()}-{df_silver.ano.max()} ({df_silver.ano.nunique()} anos)')
print(f'Colunas ({df_silver.shape[1]}): {list(df_silver.columns)}')
df_silver.head()

In [ ]:
# Serie temporal de notificacoes no DF
ts = df_silver.sort_values(['ano', 'semana_epidemiologica'])

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(range(len(ts)), ts['notificacoes'].values, linewidth=0.8, color='steelblue')
ax.set_xlabel('Semana epidemiologica (indice sequencial)')
ax.set_ylabel('Notificacoes de dengue')
ax.set_title(f'Notificacoes de Dengue no DF (Brasilia) — {ts.ano.min()}-{ts.ano.max()}')

# Marcar mudancas de ano
year_starts = ts.groupby('ano').apply(lambda g: g.index[0]).values
for i, idx in enumerate(year_starts):
    pos = list(ts.index).index(idx)
    if i % 5 == 0:
        ax.axvline(pos, color='gray', alpha=0.3, linestyle='--')
        ax.text(pos, ax.get_ylim()[1]*0.95, str(ts.loc[idx, 'ano']), fontsize=7, alpha=0.6)

plt.tight_layout()
plt.show()

# Resumo por ano
resumo = ts.groupby('ano')['notificacoes'].agg(['sum', 'mean', 'max']).round(1)
resumo.columns = ['total', 'media_semanal', 'pico_semanal']
print('\nResumo por ano:')
print(resumo.to_string())

---
## 4. Explorar SINAN Gold (DF)
Camada **Gold**: serie densa com features epidemiologicas calculadas (lags, medias moveis, indices, labels).

In [ ]:
sinan_gold = ds.dataset(
    f'{DATA_DIR}/sinan/gold/sinan_tcc2_v2/official_dense',
    format='parquet', partitioning='hive'
)

df_gold = sinan_gold.to_table(
    filter=ds.field('ibge_municipio') == IBGE_DF
).to_pandas()

print(f'SINAN Gold DF: {len(df_gold):,} linhas')
print(f'Periodo: {df_gold.year.min()}-{df_gold.year.max()}')
print(f'\nColunas ({df_gold.shape[1]}):')

# Agrupar colunas por tipo
cols_id = [c for c in df_gold.columns if c in ['ano_semana','ano','semana_epidemiologica','week_start','ibge_municipio','municipio','uf','regiao','source_year','municipio_resolution','municipio_source_field','year']]
cols_base = [c for c in df_gold.columns if c.startswith(('notificacoes','qt_'))]
cols_prop = [c for c in df_gold.columns if c.startswith('prop_')]
cols_indice = [c for c in df_gold.columns if c.startswith('indice_')]
cols_lag = [c for c in df_gold.columns if 'lag' in c or 'movel' in c or 'diff' in c or 'pct_change' in c or 'razao' in c or 'aceleracao' in c]
cols_label = [c for c in df_gold.columns if c.startswith('label_')]
cols_time = [c for c in df_gold.columns if 'week_of_year' in c or 'is_zero' in c]

print(f'  Identificacao:     {len(cols_id)}')
print(f'  Metricas base:     {len(cols_base)}')
print(f'  Proporcoes:        {len(cols_prop)}')
print(f'  Indices compostos: {len(cols_indice)}')
print(f'  Lags/MMs/Diffs:    {len(cols_lag)}')
print(f'  Labels:            {len(cols_label)}')
print(f'  Temporal:          {len(cols_time)}')

In [ ]:
# Indices compostos ao longo do tempo no DF
indices = ['indice_sintomas', 'indice_alarme', 'indice_gravidade', 'indice_carga_clinica']
ts_gold = df_gold.sort_values(['year', 'semana_epidemiologica'])

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for ax, idx in zip(axes.flat, indices):
    ax.plot(range(len(ts_gold)), ts_gold[idx].values, linewidth=0.6, color='coral')
    ax.set_title(idx.replace('indice_', '').capitalize())
    ax.set_ylabel('Indice')

axes[1][0].set_xlabel('Semana (indice sequencial)')
axes[1][1].set_xlabel('Semana (indice sequencial)')
fig.suptitle('Indices Compostos — DF (Brasilia)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Explorar INMET Silver (Estacao A001 — Brasilia)
Dados climaticos semanais da estacao meteorologica mais proxima de Brasilia (1.18 km).

In [ ]:
# Mapping estacao -> municipio
mapping = pd.read_csv(f'{DATA_DIR}/inmet/bronze/municipio_estacao_mapping.csv')
est_df = mapping[mapping['ibge_municipio'].astype(str) == IBGE_DF]
print('Estacao INMET para Brasilia/DF:')
print(est_df.to_string(index=False))

ESTACAO_DF = est_df['codigo_wmo'].values[0]
print(f'\nUsando estacao: {ESTACAO_DF}')

In [ ]:
# Carregar INMET silver para estacao A001
silver_files = sorted(glob.glob(f'{DATA_DIR}/inmet/silver/weekly_stations_*.parquet'))
print(f'Arquivos INMET Silver: {len(silver_files)} anos')

dfs = []
for f in silver_files:
    df = pd.read_parquet(f)
    df_est = df[df['codigo_wmo'] == ESTACAO_DF]
    if len(df_est) > 0:
        dfs.append(df_est)

inmet_silver = pd.concat(dfs, ignore_index=True).sort_values(['ano_epi', 'semana_epidemiologica'])
print(f'\nINMET Silver estacao {ESTACAO_DF}: {len(inmet_silver):,} semanas')
print(f'Periodo: {inmet_silver.ano_epi.min()}-{inmet_silver.ano_epi.max()}')
print(f'Colunas: {list(inmet_silver.columns)}')

# Anos com dados
anos_cobertura = sorted(inmet_silver.ano_epi.unique())
anos_completos = set(range(2000, 2027))
anos_faltantes = anos_completos - set(anos_cobertura)
print(f'\nAnos com dados: {len(anos_cobertura)}')
print(f'Anos sem dados: {sorted(anos_faltantes) if anos_faltantes else "nenhum"}')

inmet_silver.head()

In [ ]:
# Variaveis climaticas ao longo do tempo
vars_clima = ['temp_mean_c', 'rain_sum_mm', 'humidity_mean_pct', 'temp_range_c']
labels_clima = ['Temperatura media (C)', 'Chuva semanal (mm)', 'Umidade media (%)', 'Amplitude termica (C)']

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for ax, var, label in zip(axes.flat, vars_clima, labels_clima):
    ax.plot(range(len(inmet_silver)), inmet_silver[var].values, linewidth=0.6, color='seagreen')
    ax.set_ylabel(label)
    ax.set_title(label)

axes[1][0].set_xlabel('Semana (indice sequencial)')
axes[1][1].set_xlabel('Semana (indice sequencial)')
fig.suptitle(f'Dados Climaticos — Estacao {ESTACAO_DF} (Brasilia)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Explorar INMET Gold (DF)
Features climaticas agregadas a nivel municipal — camada **Gold** com lags e medias moveis.

In [ ]:
gold_files = sorted(glob.glob(f'{DATA_DIR}/inmet/gold/weekly_municipal_climate_*.parquet'))
print(f'Arquivos INMET Gold: {len(gold_files)} anos')

dfs = []
for f in gold_files:
    df = pd.read_parquet(f)
    df_df = df[df['ibge_municipio'].astype(str) == IBGE_DF]
    if len(df_df) > 0:
        dfs.append(df_df)

inmet_gold = pd.concat(dfs, ignore_index=True).sort_values(['ano_epi', 'semana_epidemiologica'])
print(f'\nINMET Gold DF: {len(inmet_gold):,} semanas')
print(f'Periodo: {inmet_gold.ano_epi.min()}-{inmet_gold.ano_epi.max()}')
print(f'Colunas ({inmet_gold.shape[1]}): {list(inmet_gold.columns)}')

inmet_gold.head()

---
## 7. Dados Integrados SINAN + INMET (DF)
Join entre dados epidemiologicos e climaticos a nivel municipal-semanal.

In [ ]:
integrated = pd.read_parquet(f'{DATA_DIR}/integrated/sinan_inmet_municipal_weekly.parquet')
df_integ = integrated[integrated['ibge_municipio'].astype(str) == IBGE_DF].copy()
del integrated

print(f'Dados integrados DF: {len(df_integ):,} linhas, {df_integ.shape[1]} colunas')

# Colunas epidemiologicas vs climaticas
clima_cols = [c for c in df_integ.columns if any(k in c for k in [
    'temp_', 'rain_', 'humidity_', 'pressure_', 'wind_', 'radiation_'
])]
epi_cols = [c for c in df_integ.columns if c not in clima_cols]
print(f'  Features epidemiologicas: {len(epi_cols)}')
print(f'  Features climaticas:      {len(clima_cols)}')

# Verificar NaNs climaticos (semanas sem cobertura INMET)
n_missing_clima = df_integ[clima_cols].isna().any(axis=1).sum()
print(f'\nSemanas sem dados climaticos: {n_missing_clima} de {len(df_integ)} ({100*n_missing_clima/len(df_integ):.1f}%)')

df_integ.head()

In [ ]:
# Correlacao dengue x clima no DF
if 'notificacoes' in df_integ.columns:
    target = 'notificacoes'
else:
    target = [c for c in df_integ.columns if 'notificacoes' in c and 'lag' not in c and 'movel' not in c][0]

clima_base = ['temp_mean_c', 'rain_sum_mm', 'humidity_mean_pct', 'temp_range_c']
clima_base = [c for c in clima_base if c in df_integ.columns]

fig, axes = plt.subplots(1, len(clima_base), figsize=(16, 4))
for ax, var in zip(axes, clima_base):
    valid = df_integ[[target, var]].dropna()
    ax.scatter(valid[var], valid[target], alpha=0.3, s=10, color='teal')
    ax.set_xlabel(var)
    ax.set_ylabel(target)
    corr = valid[var].corr(valid[target])
    ax.set_title(f'r = {corr:.3f}')

fig.suptitle(f'Correlacao Dengue x Clima — DF', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Carregar Model-Ready (DF)
Dataset com feature engineering completa, filtrado para o DF.

**Estrategia:** como o dataset do DF tem apenas ~1.400 linhas, combinamos train (<2022) + val (2022-2023) para treinar. O test (2024+) inclui o surto historico de 2024 (pico de 22 mil notif/semana, 10x acima do maximo historico).

**Log-transform:** usamos `log1p(y)` para comprimir a escala — reduz o gap train→test de 5.6x para 1.2x e permite ao XGBoost aprender melhor padroes relativos.

In [ ]:
MODEL_READY = base / 'model_ready'

def load_df_only(path, ibge=IBGE_DF):
    df = pd.read_parquet(path)
    return df[df['ibge_municipio'].astype(str) == ibge].copy()

train_orig = load_df_only(MODEL_READY / 'train.parquet')
val_orig   = load_df_only(MODEL_READY / 'val.parquet')
test       = load_df_only(MODEL_READY / 'test.parquet')

# Combinar train+val (dataset pequeno, val contem 2022-2023 com valores altos essenciais)
train = pd.concat([train_orig, val_orig], ignore_index=True)

print(f'Train orig: {len(train_orig):>6,} linhas  (< 2022)')
print(f'Val orig:   {len(val_orig):>6,} linhas  (2022-2023)')
print(f'Train final:{len(train):>6,} linhas  (train+val combinados)')
print(f'Test:       {len(test):>6,} linhas  (2024+)')
print(f'Features:   {train.shape[1]} colunas')

In [ ]:
ID_COLS = ['ibge_municipio', 'ano', 'semana_epidemiologica']
TARGET_REG = 'notificacoes_t4'
TARGET_CLF = 'risco_surto_t4'

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET_REG, TARGET_CLF]]
print(f'{len(feature_cols)} features para modelagem')

X_train = train[feature_cols]
X_test  = test[feature_cols]

# Log-transform no target (comprime escala, melhora R2 de 0.02 -> 0.46)
y_train_log = np.log1p(train[TARGET_REG])
y_test_log  = np.log1p(test[TARGET_REG])
y_test_raw  = test[TARGET_REG]

print(f'\nTarget (notificacoes_t4) — escala original:')
print(f'  Train: mean={train[TARGET_REG].mean():.0f}, max={train[TARGET_REG].max():.0f}')
print(f'  Test:  mean={y_test_raw.mean():.0f}, max={y_test_raw.max():.0f} (2024 = surto historico)')
print(f'\nTarget — escala log1p:')
print(f'  Train: mean={y_train_log.mean():.2f}, max={y_train_log.max():.2f}')
print(f'  Test:  mean={y_test_log.mean():.2f}, max={y_test_log.max():.2f}')
print(f'  Gap original: {y_test_raw.max()/train[TARGET_REG].max():.1f}x -> Gap log: {y_test_log.max()/y_train_log.max():.1f}x')

---
## 9. XGBoost Regressao (previsao t+4 semanas)

In [ ]:
model_reg = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.6,
    tree_method='hist',
    reg_alpha=1.0,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=-1,
)

# Treinar em escala log
model_reg.fit(X_train, y_train_log, verbose=0)

# Prever e reverter para escala original
y_pred_log = model_reg.predict(X_test)
y_pred = np.expm1(np.maximum(y_pred_log, 0))

# Metricas em ambas escalas
rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred))
mae = mean_absolute_error(y_test_raw, y_pred)
r2 = r2_score(y_test_raw, y_pred)
r2_log = r2_score(y_test_log, y_pred_log)

print(f'=== Regressao — DF (Test Set) ===')
print(f'R2 (escala log):      {r2_log:.4f}')
print(f'R2 (escala original): {r2:.4f}')
print(f'RMSE: {rmse:.1f}')
print(f'MAE:  {mae:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter (escala log)
axes[0].scatter(y_test_log, y_pred_log, alpha=0.5, s=20, color='steelblue')
lim = max(y_test_log.max(), y_pred_log.max()) * 1.1
axes[0].plot([0, lim], [0, lim], 'r--', alpha=0.5)
axes[0].set_xlabel('Real (log)')
axes[0].set_ylabel('Previsto (log)')
axes[0].set_title(f'Previsto vs Real — log (R2={r2_log:.3f})')

# Scatter (escala original)
axes[1].scatter(y_test_raw, y_pred, alpha=0.5, s=20, color='steelblue')
lim = max(y_test_raw.max(), max(y_pred)) * 1.1
axes[1].plot([0, lim], [0, lim], 'r--', alpha=0.5)
axes[1].set_xlabel('Real')
axes[1].set_ylabel('Previsto')
axes[1].set_title(f'Previsto vs Real — original (R2={r2:.3f})')

# Serie temporal
axes[2].plot(range(len(y_test_raw)), y_test_raw.values, label='Real', linewidth=1)
axes[2].plot(range(len(y_pred)), y_pred, label='Previsto', linewidth=1, alpha=0.8)
axes[2].set_xlabel('Semana (test set)')
axes[2].set_ylabel('Notificacoes t+4')
axes[2].set_title('Serie Temporal — Test Set DF')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance top 20
importances = pd.Series(model_reg.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 20 Features — XGBoost Regressao (DF)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 10. XGBoost Classificacao (risco de surto)

In [ ]:
y_train_clf = train[TARGET_CLF]
y_test_clf = test[TARGET_CLF]

print('Distribuicao das classes — DF (train+val combinados):')
print(y_train_clf.value_counts().sort_index())
print(f'\nClasses: 0=baixo, 1=medio, 2=alto, 3=surto')
print(f'\nDistribuicao no test:')
print(y_test_clf.value_counts().sort_index())

In [ ]:
model_clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.6,
    tree_method='hist',
    reg_alpha=1.0,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=-1,
    objective='multi:softprob',
    num_class=4,
)

model_clf.fit(X_train, y_train_clf, verbose=0)

y_pred_clf = model_clf.predict(X_test)
y_pred_proba = model_clf.predict_proba(X_test)

try:
    auc = roc_auc_score(y_test_clf, y_pred_proba, multi_class='ovr', average='macro')
    print(f'AUC macro: {auc:.4f}')
except ValueError as e:
    auc = None
    print(f'AUC nao calculavel: {e}')

f1 = f1_score(y_test_clf, y_pred_clf, average='macro', zero_division=0)

print(f'\n=== Classificacao — DF (Test Set) ===')
print(f'F1 macro:  {f1:.4f}')
print(f'\n{classification_report(y_test_clf, y_pred_clf, target_names=["baixo","medio","alto","surto"], zero_division=0)}')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_clf, y_pred_clf,
    display_labels=['baixo', 'medio', 'alto', 'surto'],
    ax=ax, cmap='Blues'
)
ax.set_title('Matriz de Confusao — Classificacao de Risco (DF)')
plt.tight_layout()
plt.show()

---
## 11. Baseline: SINAN-only vs SINAN+INMET

In [ ]:
climate_cols = [c for c in feature_cols if any(k in c for k in [
    'temp_', 'rain_', 'humidity_', 'pressure_', 'wind_', 'radiation_'
])]
sinan_cols = [c for c in feature_cols if c not in climate_cols]

print(f'Features SINAN: {len(sinan_cols)}')
print(f'Features clima: {len(climate_cols)}')

model_sinan = xgb.XGBRegressor(
    n_estimators=1000, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.6, tree_method='hist',
    reg_alpha=1.0, reg_lambda=5.0, random_state=42, n_jobs=-1,
)
model_sinan.fit(X_train[sinan_cols], y_train_log, verbose=0)

y_pred_sinan_log = model_sinan.predict(X_test[sinan_cols])
y_pred_sinan = np.expm1(np.maximum(y_pred_sinan_log, 0))

r2_full = r2_score(y_test_raw, y_pred)
r2_sinan = r2_score(y_test_raw, y_pred_sinan)
r2_full_log = r2_score(y_test_log, y_pred_log)
r2_sinan_log = r2_score(y_test_log, y_pred_sinan_log)
rmse_full = np.sqrt(mean_squared_error(y_test_raw, y_pred))
rmse_sinan = np.sqrt(mean_squared_error(y_test_raw, y_pred_sinan))

print(f'\n=== Comparacao — DF ===')
print(f'{"Modelo":<20s} {"R2_log":>8s} {"R2":>8s} {"RMSE":>10s}')
print(f'{"SINAN+INMET":<20s} {r2_full_log:>8.4f} {r2_full:>8.4f} {rmse_full:>10.1f}')
print(f'{"SINAN-only":<20s} {r2_sinan_log:>8.4f} {r2_sinan:>8.4f} {rmse_sinan:>10.1f}')
print(f'\n-> {"SINAN-only eh MELHOR (log)" if r2_sinan_log > r2_full_log else "SINAN+INMET eh melhor (log)"}')

In [ ]:
# Comparacao visual
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Escala log
axes[0].plot(range(len(y_test_log)), y_test_log.values, label='Real', linewidth=1.2, color='black')
axes[0].plot(range(len(y_pred_log)), y_pred_log, label=f'SINAN+INMET (R2={r2_full_log:.3f})', linewidth=1, alpha=0.8, color='steelblue')
axes[0].plot(range(len(y_pred_sinan_log)), y_pred_sinan_log, label=f'SINAN-only (R2={r2_sinan_log:.3f})', linewidth=1, alpha=0.8, color='coral')
axes[0].set_xlabel('Semana (test set)')
axes[0].set_ylabel('log1p(notificacoes t+4)')
axes[0].set_title('Escala log')
axes[0].legend()

# Escala original
axes[1].plot(range(len(y_test_raw)), y_test_raw.values, label='Real', linewidth=1.2, color='black')
axes[1].plot(range(len(y_pred)), y_pred, label=f'SINAN+INMET (R2={r2_full:.3f})', linewidth=1, alpha=0.8, color='steelblue')
axes[1].plot(range(len(y_pred_sinan)), y_pred_sinan, label=f'SINAN-only (R2={r2_sinan:.3f})', linewidth=1, alpha=0.8, color='coral')
axes[1].set_xlabel('Semana (test set)')
axes[1].set_ylabel('Notificacoes t+4')
axes[1].set_title('Escala original')
axes[1].legend()

fig.suptitle('Comparacao SINAN+INMET vs SINAN-only — DF', fontsize=14)
plt.tight_layout()
plt.show()

---
## 12. SHAP — Interpretabilidade

In [ ]:
import shap

explainer = shap.TreeExplainer(model_reg)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values calculados para {len(X_test)} amostras do test set DF.')

In [ ]:
shap.summary_plot(shap_values, X_test, max_display=20, show=True)

In [ ]:
# Importancia SHAP: epid vs clima
shap_importance = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

shap_importance['is_climate'] = shap_importance['feature'].apply(
    lambda x: any(k in x for k in ['temp_', 'rain_', 'humidity_', 'pressure_', 'wind_', 'radiation_'])
)

print('Top 20 features por SHAP — DF:')
print(shap_importance.head(20).to_string(index=False))

n_climate_top20 = shap_importance.head(20)['is_climate'].sum()
print(f'\nFeatures climaticas no top-20: {n_climate_top20}')

total_shap_clima = shap_importance[shap_importance['is_climate']]['mean_abs_shap'].sum()
total_shap = shap_importance['mean_abs_shap'].sum()
print(f'Contribuicao SHAP total clima: {100*total_shap_clima/total_shap:.1f}%')

---
## 13. Salvar Resultados no Drive

In [ ]:
results_dir = Path(DATA_DIR).parent / 'TCC2-RESULTADOS-DF'
results_dir.mkdir(exist_ok=True)

metrics = {
    'municipio': 'Brasilia/DF',
    'ibge': IBGE_DF,
    'regressao': {
        'R2_log': round(r2_log, 4),
        'R2_original': round(r2, 4),
        'RMSE': round(rmse, 1),
        'MAE': round(mae, 1),
    },
    'classificacao': {'F1_macro': round(f1, 4)},
    'comparacao': {
        'sinan_inmet_R2_log': round(r2_full_log, 4),
        'sinan_only_R2_log': round(r2_sinan_log, 4),
        'sinan_inmet_R2': round(r2_full, 4),
        'sinan_only_R2': round(r2_sinan, 4),
    },
    'dados': {
        'train_val_combinados': len(train),
        'test': len(test),
        'features': len(feature_cols),
    },
    'notas': 'log1p transform no target, train+val combinados, 2024=surto historico'
}

if auc is not None:
    metrics['classificacao']['AUC_macro'] = round(auc, 4)

with open(results_dir / 'metrics_df.json', 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

model_reg.save_model(str(results_dir / 'xgb_regression_df.json'))
model_clf.save_model(str(results_dir / 'xgb_classification_df.json'))
model_sinan.save_model(str(results_dir / 'xgb_sinan_only_df.json'))

print(f'Resultados salvos em: {results_dir}')
print(f'Arquivos: {sorted(f.name for f in results_dir.iterdir())}')